# HW07: Transformers and Doc Embeddings

Remember that these homework work as a completion grade. **You can skip one section of this homework.**

In [ ]:
!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df = df.sample(n=10000) # # only use 10K datapoints
df.head()

,label,title,lead,text
7946,sport,Ex-Hornets Executive Sentenced for Fraud (AP),"AP - Spencer Stolpen, former president of the ...",Ex-Hornets Executive Sentenced for Fraud (AP) ...
98024,sport,England: Balckburn still in last,Blackburn recovered from 3-1 down to draw 3-3 ...,England: Balckburn still in last Blackburn rec...
44953,business,Microsoft Asks EU Court to Suspend Antitrust R...,"Microsoft Corp., the world #39;s largest softw...",Microsoft Asks EU Court to Suspend Antitrust R...
81547,world,Seoul worried about Bush #39;s N. Korea stance,South Korean officials were very cautious when...,Seoul worried about Bush #39;s N. Korea stance...
23020,business,Dollar Supported by Dip in Jobless Claims,CHICAGO (Reuters) - The dollar posted mild ga...,Dollar Supported by Dip in Jobless Claims CHI...


## Hugginface Transformers

In [ ]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig
import torch  # fixed typo (was "troch")

In [ ]:
##TODO build a transformer model to do sequence classification with the goal to predict the label from the text

# Map the 4 human-readable class names to integer indices and back.
# DistilBERT's classification head outputs one logit per class, so we need integers internally.
label2id = {"world": 0, "sport": 1, "business": 2, "sci/tech": 3}
id2label  = {v: k for k, v in label2id.items()}

# Load a pre-trained DistilBERT checkpoint and attach a 4-way classification head on top.
# "distilbert-base-uncased" is a smaller, faster version of BERT (6 layers, 66M params).
# num_labels=4 tells HuggingFace to add a linear layer that maps [CLS] → 4 class scores.
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4,
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
##TODO print the summary of the model

# Print the full layer-by-layer architecture so you can see the encoder stack
# and the classification head (pre_classifier + classifier) that was added.
print(model)

# Also report parameter counts — useful to know how much the GPU needs to hold.
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
##TODO split the sample into a training and a test set
##TODO prepare the dataset for torch.

from transformers import DistilBertTokenizer
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

# Load the tokenizer that pairs with our model checkpoint.
# It converts raw strings → token IDs, attention masks, etc.
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# 80/20 train/test split; stratify=df["label"] keeps class proportions balanced.
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")


class NewsDataset(Dataset):
    """Tokenises text at construction time and stores tensors ready for the DataLoader."""

    def __init__(self, texts, labels, tokenizer, max_length=128):
        # Tokenize the entire split in one batch call (fast).
        # truncation=True  → clips sequences longer than max_length.
        # padding="max_length" → pads shorter sequences so every tensor is the same size.
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        # Convert string labels ("world", "sport", …) to integer class indices.
        self.labels = torch.tensor([label2id[l] for l in labels])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Return the three tensors the Trainer expects for each example.
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }


train_dataset = NewsDataset(train_df["text"], train_df["label"], tokenizer)
test_dataset  = NewsDataset(test_df["text"],  test_df["label"],  tokenizer)
print("Datasets ready.")

In [ ]:
##TODO fit the model and print the obtained accuracy
##       (hint: you can follow the training steps in the notebook.
##        To learn more, checkout the trainer class of huggingface transformers)

import numpy as np
from transformers import TrainingArguments, Trainer
import evaluate

# Load the HuggingFace accuracy metric (equivalent to sklearn accuracy_score).
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Called automatically by Trainer after every evaluation pass."""
    logits, labels = eval_pred
    # argmax picks the class with the highest raw score (logit) for each example.
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


# TrainingArguments is a dataclass that controls every hyperparameter and I/O path.
training_args = TrainingArguments(
    output_dir="./results",           # where checkpoints and logs are written
    num_train_epochs=3,               # full passes through the training set
    per_device_train_batch_size=16,   # examples per GPU step during training
    per_device_eval_batch_size=32,    # larger batch is fine for eval (no gradients)
    warmup_steps=100,                 # linearly ramp the LR up for the first 100 steps
    weight_decay=0.01,                # L2 penalty to help prevent overfitting
    eval_strategy="epoch",            # evaluate on test set at the end of each epoch
    save_strategy="epoch",            # save a checkpoint at the end of each epoch
    load_best_model_at_end=True,      # restore the best checkpoint when training finishes
    logging_steps=50,                 # print a training log line every 50 steps
)

# Trainer orchestrates the training loop, gradient updates, evaluation, and checkpointing.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Fine-tune DistilBERT on the AG News training split.
# Per-epoch accuracy is printed automatically by the Trainer.
trainer.train()

# Run a final evaluation pass on the held-out test set.
results = trainer.evaluate()
print(f"\nFinal test accuracy: {results['eval_accuracy']:.4f}")

# Doc Embedding

In [ ]:
# obtain the data
!wget http://alt.qcri.org/semeval2017/task1/data/uploads/sts2017.eval.v1.1.zip
!wget http://alt.qcri.org/semeval2017/task1/data/uploads/sts2017.gs.zip

!unzip sts2017.eval.v1.1.zip 
!unzip sts2017.gs.zip 

In [ ]:
# load the data

def load_STS_data():
    with open("STS2017.gs/STS.gs.track5.en-en.txt") as f:
        labels = [float(line.strip()) for line in f]
    
    text_a, text_b = [], []
    with open("STS2017.eval.v1.1/STS.input.track5.en-en.txt") as f:
        for line in f:
            line = line.strip().split("\t")
            text_a.append(line[0])
            text_b.append(line[1])
    return text_a, text_b, labels

text_a, text_b, labels = load_STS_data()
text_a[0], text_b[0], labels[0]

In [ ]:
# some utils
from scipy.stats import spearmanr
def evaluate(predictions, labels):
    print ("spearman's rank correlation", spearmanr(predictions, labels)[0])

import numpy as np
from numpy import dot
from numpy.linalg import norm

def cosine_similarity(a,b):
    return dot(a, b)/(norm(a)*norm(b))


In [ ]:
# Wordcounts baseline
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer()
vec.fit(text_a + text_b)

# encode documents
text_a_encoded = np.array(vec.transform(text_a).todense())
text_b_encoded = np.array(vec.transform(text_b).todense())

# predict cosine similarities
predictions = [cosine_similarity(a,b) for a,b in zip(text_a_encoded, text_b_encoded)]

# evaluate
evaluate(predictions, labels)

In [ ]:
##TODO train Doc2Vec on the texts in the dataset
##TODO derive the word vectors for each text in the dataset
##TODO compute cosine similarity between the text pairs and evaluate spearman's rank correlation
## Don't worry if results are not satisfactory using Doc2Vec (the dataset is too small to train good embeddings)

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import nltk
nltk.download("punkt_tab")

# Doc2Vec requires each document to be tokenised into a list of words and given a unique tag.
# TaggedDocument pairs the word list with an integer tag so the model knows which vector = which doc.
all_texts = text_a + text_b
tagged_docs = [
    TaggedDocument(words=nltk.word_tokenize(doc.lower()), tags=[i])
    for i, doc in enumerate(all_texts)
]

# Train a PV-DM (default) Doc2Vec model.
# vector_size: dimensionality of each document embedding
# window:      context window around each word during training
# min_count:   ignore words that appear fewer than this many times
# epochs:      training passes over the corpus
doc2vec_model = Doc2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    epochs=40,
    workers=4,
)
doc2vec_model.build_vocab(tagged_docs)   # build the vocabulary from all documents
doc2vec_model.train(                     # train the model
    tagged_docs,
    total_examples=doc2vec_model.corpus_count,
    epochs=doc2vec_model.epochs,
)

# Infer a fixed-size vector for each text in the STS dataset.
# gensim 4.x renamed the 'steps' argument to 'epochs' — use epochs=20 here.
text_a_vecs = [doc2vec_model.infer_vector(nltk.word_tokenize(t.lower()), epochs=20) for t in text_a]
text_b_vecs = [doc2vec_model.infer_vector(nltk.word_tokenize(t.lower()), epochs=20) for t in text_b]

# Compute cosine similarity for each sentence pair, then evaluate with Spearman's ρ.
doc2vec_predictions = [cosine_similarity(a, b) for a, b in zip(text_a_vecs, text_b_vecs)]
print("Doc2Vec results:")
evaluate(doc2vec_predictions, labels)

In [ ]:
##TODO do the same with embeddings provided by spaCy

import spacy

# en_core_web_md includes 300-dim GloVe word vectors (en_core_web_sm does not have vectors).
# Download once with: python -m spacy download en_core_web_md
!python -m spacy download en_core_web_md

nlp = spacy.load("en_core_web_md")

# spaCy's doc.vector is the mean of all token vectors in the document.
# disable=["ner","parser"] skips components we don't need, making it faster.
text_a_spacy = [nlp(t, disable=["ner", "parser"]).vector for t in text_a]
text_b_spacy = [nlp(t, disable=["ner", "parser"]).vector for t in text_b]

# Compute cosine similarity for each pair and evaluate with Spearman's ρ.
spacy_predictions = [cosine_similarity(a, b) for a, b in zip(text_a_spacy, text_b_spacy)]
print("spaCy results:")
evaluate(spacy_predictions, labels)

In [ ]:
##TODO do the same with SBERT embeddings

from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2 is a lightweight SBERT model fine-tuned specifically for
# semantic similarity tasks — fast on GPU and strong on STS benchmarks.
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

# encode() tokenises and embeds an entire list in one batched forward pass.
# show_progress_bar gives a live update since this takes a few seconds.
text_a_sbert = sbert_model.encode(text_a, show_progress_bar=True)
text_b_sbert = sbert_model.encode(text_b, show_progress_bar=True)

# Compute cosine similarity for each pair and evaluate with Spearman's ρ.
sbert_predictions = [cosine_similarity(a, b) for a, b in zip(text_a_sbert, text_b_sbert)]
print("SBERT results:")
evaluate(sbert_predictions, labels)